In [21]:
!pip install -q google-genai faiss-cpu pandas numpy

In [22]:
import os
import json
import zipfile
import numpy as np
import pandas as pd
import faiss

from google import genai
from google.colab import userdata

print("Libraries imported successfully")

Libraries imported successfully


In [23]:
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY not found")

print("API key loaded successfully")

API key loaded successfully


In [24]:
import os

print(os.listdir("/content"))

['.config', 'jobs_data', 'sample_data', 'naukri_com-job_sample.csv.zip']


In [25]:
import zipfile
import os

zip_path = "/content/naukri_com-job_sample.csv.zip"
extract_folder = "/content/jobs_data"

os.makedirs(extract_folder, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_folder)

print("ZIP extracted successfully!")
print(os.listdir(extract_folder))

ZIP extracted successfully!
['naukri_com-job_sample.csv']


In [26]:
import pandas as pd

csv_path = "/content/jobs_data/naukri_com-job_sample.csv"

jobs_df = pd.read_csv(csv_path, encoding="latin1")

print("Jobs loaded successfully!")
print("Number of rows:", len(jobs_df))
print("Number of columns:", len(jobs_df.columns))

jobs_df.head()

Jobs loaded successfully!
Number of rows: 22000
Number of columns: 14


,company,education,experience,industry,jobdescription,jobid,joblocation_address,jobtitle,numberofpositions,payrate,postdate,site_name,skills,uniq_id
0,MM Media Pvt Ltd,UG: B.Tech/B.E. - Any Specialization PG:Any Po...,0 - 1 yrs,Media / Entertainment / Internet,Job Description Â Send me Jobs like this Qual...,210516002263,Chennai,Walkin Data Entry Operator (night Shift),NaN,"1,50,000 - 2,25,000 P.A",2016-05-21 19:30:00 +0000,NaN,ITES,43b19632647068535437c774b6ca6cf8
1,find live infotech,UG: B.Tech/B.E. - Any Specialization PG:MBA/PG...,0 - 0 yrs,Advertising / PR / MR / Event Management,Job Description Â Send me Jobs like this Qual...,210516002391,Chennai,Work Based Onhome Based Part Time.,60.0,"1,50,000 - 2,50,000 P.A. 20000",2016-05-21 19:30:00 +0000,NaN,Marketing,d4c72325e57f89f364812b5ed5a795f0
2,Softtech Career Infosystem Pvt. Ltd,UG: Any Graduate - Any Specialization PG:Any P...,4 - 8 yrs,IT-Software / Software Services,Job Description Â Send me Jobs like this - as...,101016900534,Bengaluru,Pl/sql Developer - SQL,NaN,Not Disclosed by Recruiter,2016-10-13 16:20:55 +0000,NaN,IT Software - Application Programming,c47df6f4cfdf5b46f1fd713ba61b9eba
3,Onboard HRServices LLP,UG: Any Graduate - Any Specialization PG:CA Do...,11 - 15 yrs,Banking / Financial Services / Broking,Job Description Â Send me Jobs like this - In...,81016900536,"Mumbai, Bengaluru, Kolkata, Chennai, Coimbator...",Manager/ad/partner - Indirect Tax - CA,NaN,Not Disclosed by Recruiter,2016-10-13 16:20:55 +0000,NaN,Accounts,115d28f140f694dd1cc61c53d03c66ae
4,Spire Technologies and Solutions Pvt. Ltd.,UG: B.Tech/B.E. - Any Specialization PG:Any Po...,6 - 8 yrs,IT-Software / Software Services,Job Description Â Send me Jobs like this Plea...,120916002122,Bengaluru,JAVA Technical Lead (6-8 yrs) -,4.0,Not Disclosed by Recruiter,2016-10-13 16:20:55 +0000,NaN,IT Software - Application Programming,a12553fc03bc7bcced8b1bb8963f97b4


In [13]:
print(jobs_df.columns.tolist())

['company', 'education', 'experience', 'industry', 'jobdescription', 'jobid', 'joblocation_address', 'jobtitle', 'numberofpositions', 'payrate', 'postdate', 'site_name', 'skills', 'uniq_id']


In [27]:
jobs_df["combined_text"] = (
    jobs_df["jobtitle"].fillna("").astype(str)
    + " | Skills: "
    + jobs_df["skills"].fillna("").astype(str)
    + " | Description: "
    + jobs_df["jobdescription"].fillna("").astype(str)
)

print("Combined job text created successfully!")
print(jobs_df["combined_text"].iloc[0])

Combined job text created successfully!
Walkin Data Entry Operator (night Shift) | Skills: ITES | Description: Job Description Â  Send me Jobs like this Qualifications: - == > 10th To Graduation & Any Skill: - == > Basic Computer Knowledge Job Requirement : - == > System or Laptop Type of job: - == > Full Time or Part time Languages : - == > Tamil & English. Experience : - == > Freshers & Experience payment details: - 1 form per day 5/- 10 form per day 50/- 100 form per day 500/- monthly you can earn 15000/- per month Selection Process: - == > Easy Selection Process,So What Are You Waiting For? Apply Now & Grab Best Opportunity To Make Your Carrier & To Improve Your Earing Skills. More detail contact Mr Hari 8678902528 9003010282 Salary:INR 1,50,000 - 2,25,000 P.A Industry: Media / Entertainment / Internet Functional Area: ITES , BPO , KPO , LPO , Customer Service , Operations Role Category:Other Role:Fresher Keyskills English Typing Part Time Data Entry Selection Process Desired Candi

In [28]:
print("Total jobs:", len(jobs_df))
print("Missing combined text:", jobs_df["combined_text"].isna().sum())

jobs_df[["jobtitle", "skills", "jobdescription", "combined_text"]].head()

Total jobs: 22000
Missing combined text: 0


,jobtitle,skills,jobdescription,combined_text
0,Walkin Data Entry Operator (night Shift),ITES,Job Description Â Send me Jobs like this Qual...,Walkin Data Entry Operator (night Shift) | Ski...
1,Work Based Onhome Based Part Time.,Marketing,Job Description Â Send me Jobs like this Qual...,Work Based Onhome Based Part Time. | Skills: M...
2,Pl/sql Developer - SQL,IT Software - Application Programming,Job Description Â Send me Jobs like this - as...,Pl/sql Developer - SQL | Skills: IT Software -...
3,Manager/ad/partner - Indirect Tax - CA,Accounts,Job Description Â Send me Jobs like this - In...,Manager/ad/partner - Indirect Tax - CA | Skill...
4,JAVA Technical Lead (6-8 yrs) -,IT Software - Application Programming,Job Description Â Send me Jobs like this Plea...,JAVA Technical Lead (6-8 yrs) - | Skills: IT S...


In [29]:
!pip install -q google-genai faiss-cpu

In [30]:
import faiss
import numpy as np
from google import genai

print("FAISS and Gemini imported successfully!")

FAISS and Gemini imported successfully!


In [31]:
client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini client created successfully!")

Gemini client created successfully!


In [32]:
test_text = jobs_df["combined_text"].iloc[0]

result = client.models.embed_content(
    model="gemini-embedding-001",
    contents=test_text
)

embedding = np.array(result.embeddings[0].values, dtype="float32")

print("Embedding created successfully!")
print("Embedding dimension:", len(embedding))

Embedding created successfully!
Embedding dimension: 3072
